# 미국 수출 데이터 수집 V3

## V2 → V3 변경 사항

### 1) 연례 개정(annual revision) 반영
Census는 매년 최근 3개년치 데이터를 개정 발표합니다. V2는 `INSERT IGNORE` 방식이라
이미 저장된 `(hs_code, date)` 조합은 절대 갱신되지 않아, 개정된 값이 반영되지 않는 문제가 있었습니다.

- `INSERT IGNORE` → `INSERT ... ON DUPLICATE KEY UPDATE` 로 변경 (UPSERT)
- `REVISION_WINDOW_YEARS`(기본 3년) 이내 데이터는 **이미 DB에 있어도** 재수집 대상에 포함
- `REVISION_TRIGGER_MONTHS`(기본 6~7월, Census 연례 개정 발표 시기)에는 자동으로 개정 재수집 모드 켜짐
- `FORCE_REVISION_REFETCH = True` 로 수동 설정하면 지금 즉시 최근 3년치를 강제 재수집

### 2) API 실패 원인 구분 및 로깅
V2는 200/429 외 모든 응답(204 진짜 데이터없음, 4xx, 5xx)을 재시도 없이 동일하게 `None` 처리해서,
실제로는 데이터가 있는데 놓친 건지 진짜 없는 건지 구분이 안 됐습니다.

- `success` / `no_data_204`(정상, 실적 없음) / `empty_200` / `client_error_*` / `server_error_*`(재시도) / `timeout` / `exception_*` 로 태그 구분
- 5xx·429·timeout·기타 예외는 재시도, 204·4xx는 재시도 없이 사유만 기록
- 실행 종료 시 상태별 건수를 로그로 출력하여 실패 원인 파악 가능

DB 스키마(테이블명, 컬럼)는 V2와 동일하며, 같은 테이블(`us_export_data`)에 이어서 저장됩니다.


In [1]:

# -*- coding: utf-8 -*-
"""
us_export_data_collect_v3.py
--------------------------
미국 Census API -> HS 코드별 수출(Export) 월별 데이터 수집 후 MySQL 저장 (V3)

V2 대비 변경사항:
    1) [연례 개정 반영] Census가 매년 최근 3개년치 데이터를 개정 발표하는 것에 대응하여
       INSERT IGNORE -> INSERT ... ON DUPLICATE KEY UPDATE 로 변경.
       REVISION_WINDOW_YEARS 기간 내 데이터는 이미 저장되어 있어도 재수집 대상에 포함,
       최신값으로 덮어씀 (강제 재수집 모드는 FORCE_REVISION_REFETCH 로 on/off 가능).
    2) [실패 원인 구분] API 응답을 성공 / 데이터없음(204) / 클라이언트 오류(4xx) /
       서버 오류(5xx, 재시도) / 타임아웃-예외 로 구분하여 로깅.
       5xx-timeout은 재시도, 204(진짜 데이터 없음)와 4xx는 재시도하지 않고 사유 기록.

폴더 구조:
    stock_forecast/
    +-- DATA/
    |   +-- config.py                  <- DB 정보/공통 설정
    |   +-- us_top_export_hs_code.py   <- HS 코드 리스트
    +-- US_Market/collect/us_trade_export_data/
        +-- us_export_data_collect_v3.py  <- 이 파일

최초 1회 패키지 설치:
    pip install aiohttp nest_asyncio tqdm pymysql sqlalchemy
"""

# -- 표준 라이브러리 --------------------------------------------------------
import asyncio
import sys
import os
import time
from collections import Counter
from pathlib import Path
from typing import List, Optional, Tuple

# -- 서드파티 ----------------------------------------------------------------
import nest_asyncio          # Jupyter 이벤트 루프 충돌 방지 (핵심 수정)
nest_asyncio.apply()         # 반드시 다른 import 보다 먼저 적용

import aiohttp
import pandas as pd
from sqlalchemy import text
from tqdm import tqdm

# ==============================================================================
# 1. 범용 경로 설정 - DATA 폴더 자동 탐색
# ==============================================================================
def setup_universal_paths() -> dict:
    current = Path.cwd()
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print("=" * 70)
            print("경로 설정 완료")
            print("=" * 70)
            print(f"  프로젝트 루트 : {parent}")
            print(f"  DATA 폴더     : {data_folder}")
            print(f"  현재 위치     : {current}")
            print(f"  운영체제      : {os.name}")
            print("=" * 70 + "\n")
            return {"project_root": parent, "data_folder": data_folder, "current": current}
    raise FileNotFoundError(
        f"DATA 폴더를 찾을 수 없습니다.\n현재 위치: {current}"
    )

try:
    paths = setup_universal_paths()
except FileNotFoundError as e:
    print(e)
    sys.exit(1)

# ==============================================================================
# 2. 프로젝트 내부 모듈 import
#    config.py 에 실제로 존재하는 이름만 사용
# ==============================================================================
from config import (
    get_db_info,        # DB 접속 정보 dict 반환
    get_engine,         # SQLAlchemy engine 생성
    log,                # [TAG] message 출력
    BATCH_SIZE_DEFAULT, # 배치 기본값 (=20)
    START_DATE_MONTH,   # 수집 기본 시작일
)
from us_top_export_hs_code import US_TOP_EXPORT_HS_CODES

# Census API 키 - config.py 에 없으므로 여기서 직접 정의
API_KEY = "bf388499b71a365d725e1c888201736f7409d7e4"

# ==============================================================================
# 3. 수집 파라미터
# ==============================================================================
COLLECT_START  = "2020-01"

# 현재 달 기준 2개월 전 (Census API 최신 데이터는 미확정이므로 제외)
COLLECT_END    = (pd.Timestamp.today() - pd.DateOffset(months=2)).strftime("%Y-%m")

TABLE_NAME     = "us_export_data"

MAX_CONCURRENT = 25      # 동시 요청 수 - Census API 안전 한도
RETRY_COUNT    = 3       # 실패 시 재시도 횟수
RETRY_DELAY    = 2.0     # 재시도 대기 (초)
DB_CHUNK_SIZE  = 1_000   # DB INSERT 청크 크기

# -- [V3 신규] 연례 개정(annual revision) 대응 파라미터 -----------------------
# Census는 매년 최근 3개년치 데이터를 개정 발표한다 (보통 4월 통계 발표 시점 = 6월경).
# INSERT IGNORE 방식으로는 이미 저장된 데이터가 절대 갱신되지 않으므로,
# 아래 옵션으로 "이미 저장돼 있어도" 최근 N년치를 재수집(UPSERT)하도록 한다.
REVISION_WINDOW_YEARS   = 3          # 개정 반영 대상 기간 (최근 N년)
REVISION_TRIGGER_MONTHS = {6, 7}     # Census 연례 개정 발표 시기 (자동 트리거 월)

# True 로 직접 켜면 지금 즉시 최근 REVISION_WINDOW_YEARS 년치를 강제 재수집한다.
# None 이면 REVISION_TRIGGER_MONTHS 에 해당하는 달에만 자동으로 켜진다.
FORCE_REVISION_REFETCH: Optional[bool] = True

_today_month = pd.Timestamp.today().month
REVISION_MODE = (
    FORCE_REVISION_REFETCH
    if FORCE_REVISION_REFETCH is not None
    else _today_month in REVISION_TRIGGER_MONTHS
)

# ==============================================================================
# 4. DB DDL / SQL
# ==============================================================================
DDL = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_NAME}` (
    `id`         BIGINT      NOT NULL AUTO_INCREMENT,
    `hs_code`    VARCHAR(10) NOT NULL  COMMENT 'HS 6자리 코드',
    `date`       DATE        NOT NULL  COMMENT '해당 월의 말일',
    `year`       CHAR(4)     NOT NULL,
    `month`      CHAR(2)     NOT NULL,
    `exp_dlr`    BIGINT               COMMENT '수출금액 (USD)',
    `created_at` DATETIME    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    `updated_at` DATETIME    NOT NULL DEFAULT CURRENT_TIMESTAMP
                                       ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (`id`),
    UNIQUE KEY `uq_hs_date` (`hs_code`, `date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
  COMMENT='미국 HS 코드별 월별 수출 데이터';
"""

# [V3 변경] INSERT IGNORE -> INSERT ... ON DUPLICATE KEY UPDATE
# 기존 값과 다를 때만 실제로 갱신되며, updated_at 이 자동으로 최신화된다.
UPSERT_SQL = f"""
INSERT INTO `{TABLE_NAME}`
    (hs_code, date, year, month, exp_dlr)
VALUES
    (:hs_code, :date, :year, :month, :exp_dlr)
ON DUPLICATE KEY UPDATE
    exp_dlr    = VALUES(exp_dlr),
    updated_at = CURRENT_TIMESTAMP
"""

# ==============================================================================
# 5. DB 유틸 함수
# ==============================================================================
def ensure_table(engine) -> None:
    """테이블이 없으면 자동 생성"""
    with engine.connect() as conn:
        conn.execute(text(DDL))
        conn.commit()
    log("DB", f"테이블 '{TABLE_NAME}' 준비 완료")


def load_existing_keys(engine) -> set:
    """이미 저장된 (hs_code, 'yyyy-mm') 조합 로드"""
    with engine.connect() as conn:
        rows = conn.execute(
            text(f"SELECT hs_code, DATE_FORMAT(date, '%Y-%m') FROM `{TABLE_NAME}`")
        ).fetchall()
    existing = {(r[0], r[1]) for r in rows}
    log("DB", f"기존 저장 건수: {len(existing):,} 개")
    return existing


def save_to_db(records: List[dict], engine) -> int:
    """UPSERT 방식 저장. 반환값: 처리 시도 건수"""
    if not records:
        return 0

    rows = []
    for r in records:
        val = r.get("exp_dlr")
        try:
            val_num = int(val) if val not in (None, "None", "") else None
            if val_num is not None and val_num > 1_000_000_000_000_000_000:
                val_num = None
        except (ValueError, TypeError):
            val_num = None

        date_obj = (
            pd.Timestamp(f"{r['year']}-{r['month']}-01") + pd.offsets.MonthEnd(0)
        ).date()

        rows.append({
            "hs_code": r["hs_code"],
            "date":    date_obj,
            "year":    r["year"],
            "month":   r["month"],
            "exp_dlr": val_num,
        })

    processed = 0
    with engine.connect() as conn:
        for i in range(0, len(rows), DB_CHUNK_SIZE):
            chunk = rows[i : i + DB_CHUNK_SIZE]
            conn.execute(text(UPSERT_SQL), chunk)
            processed += len(chunk)
        conn.commit()

    return processed


# ==============================================================================
# 6. 비동기 API 요청
# ==============================================================================
BASE_URL = "https://api.census.gov/data/timeseries/intltrade/exports/hs"


async def fetch_one(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    hs: str,
    year: str,
    month: str,
) -> Tuple[Optional[dict], str]:
    """
    반환값: (레코드 또는 None, 상태 태그)

    상태 태그:
        success        : 정상 데이터 수신
        no_data_204    : 해당 조합 실제로 무역실적 없음 (Census 공식 사양상 정상 응답)
        empty_200      : 200 인데 헤더만 온 경우 (사실상 no_data 와 동일하게 취급)
        client_error_* : 4xx (재시도 없음, 요청 자체 문제일 가능성)
        server_error_* : 5xx (재시도 후에도 실패)
        timeout        : 재시도 후에도 타임아웃
        exception_*    : 그 외 예외, 재시도 후에도 실패
        retry_exhausted: 429 재시도를 모두 소진
    """
    url = (
        f"{BASE_URL}?get=ALL_VAL_MO"
        f"&key={API_KEY}"
        f"&YEAR={year}&MONTH={month}&E_COMMODITY={hs}"
    )

    for attempt in range(1, RETRY_COUNT + 1):
        async with semaphore:
            try:
                async with session.get(
                    url, timeout=aiohttp.ClientTimeout(total=30)
                ) as resp:
                    status = resp.status

                    if status == 200:
                        data = await resp.json(content_type=None)
                        if len(data) > 1:
                            return {
                                "hs_code": hs,
                                "year":    year,
                                "month":   month,
                                "exp_dlr": data[1][0],
                            }, "success"
                        return None, "empty_200"

                    if status == 204:
                        # 공식 사양: 해당 조합에 실제 무역실적이 없는 정상 응답
                        return None, "no_data_204"

                    if status == 429:
                        # Rate limit -> 재시도
                        await asyncio.sleep(RETRY_DELAY * attempt)
                        continue

                    if 500 <= status < 600:
                        # 서버 일시 오류 -> 재시도
                        if attempt < RETRY_COUNT:
                            await asyncio.sleep(RETRY_DELAY * attempt)
                            continue
                        return None, f"server_error_{status}"

                    # 그 외 4xx -> 재시도 의미 없음, 즉시 사유 기록 후 종료
                    return None, f"client_error_{status}"

            except asyncio.TimeoutError:
                if attempt < RETRY_COUNT:
                    await asyncio.sleep(RETRY_DELAY)
                    continue
                return None, "timeout"
            except Exception as e:
                if attempt < RETRY_COUNT:
                    await asyncio.sleep(RETRY_DELAY)
                    continue
                return None, f"exception_{type(e).__name__}"

    return None, "retry_exhausted"


async def collect_all(tasks: List[tuple]) -> Tuple[List[dict], Counter]:
    """전체 태스크를 MAX_CONCURRENT 동시 요청으로 처리, 결과와 상태 통계를 함께 반환"""
    FLUSH       = 500
    semaphore   = asyncio.Semaphore(MAX_CONCURRENT)
    results     = []
    tag_counter = Counter()

    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT, ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        with tqdm(
            total=len(tasks), desc="수출 데이터 수집",
            unit="req", ncols=90, file=sys.stdout
        ) as pbar:
            for i in range(0, len(tasks), FLUSH):
                batch = tasks[i : i + FLUSH]
                coros = [fetch_one(session, semaphore, hs, yr, mo)
                         for hs, yr, mo in batch]
                batch_results = await asyncio.gather(*coros)
                for record, tag in batch_results:
                    tag_counter[tag] += 1
                    if record is not None:
                        results.append(record)
                pbar.update(len(batch))

    return results, tag_counter


# ==============================================================================
# 7. 메인
# ==============================================================================
def main():
    t0 = time.time()
    log("START", f"미국 수출 데이터 수집 시작 (V3) | {COLLECT_START} ~ {COLLECT_END}")
    log("INFO",  f"수집 대상 HS 코드: {len(US_TOP_EXPORT_HS_CODES):,} 개")
    log("INFO",  f"개정 반영 모드(REVISION_MODE): {REVISION_MODE} "
                 f"(최근 {REVISION_WINDOW_YEARS}년치 강제 재수집 여부)")

    # DB 준비
    db_info = get_db_info()
    engine  = get_engine(db_info)
    ensure_table(engine)

    # 이미 저장된 키 로드
    existing   = load_existing_keys(engine)
    date_range = pd.date_range(start=COLLECT_START, end=COLLECT_END, freq="MS")

    # 개정 반영 기준월 (오늘 기준 REVISION_WINDOW_YEARS 년 전)
    revision_cutoff = (
        pd.Timestamp.today() - pd.DateOffset(years=REVISION_WINDOW_YEARS)
    ).strftime("%Y-%m")

    # 신규 수집 대상 (기존 미보유)
    new_tasks = [
        (hs, dt.strftime("%Y"), dt.strftime("%m"))
        for hs in US_TOP_EXPORT_HS_CODES
        for dt in date_range
        if (hs, dt.strftime("%Y-%m")) not in existing
    ]

    # [V3 신규] 개정 재수집 대상 (이미 보유 중이지만 최근 N년 이내라 재확인 필요)
    revision_tasks = []
    if REVISION_MODE:
        revision_tasks = [
            (hs, dt.strftime("%Y"), dt.strftime("%m"))
            for hs in US_TOP_EXPORT_HS_CODES
            for dt in date_range
            if dt.strftime("%Y-%m") >= revision_cutoff
            and (hs, dt.strftime("%Y-%m")) in existing   # new_tasks 와 중복 방지
        ]

    tasks = new_tasks + revision_tasks
    total_possible = len(US_TOP_EXPORT_HS_CODES) * len(date_range)

    log("INFO", (
        f"전체 가능 조합: {total_possible:,} 개 | "
        f"기존 보유: {len(existing):,} 개 | "
        f"신규 수집 대상: {len(new_tasks):,} 개 | "
        f"개정 재수집 대상: {len(revision_tasks):,} 개 (기준월 >= {revision_cutoff})"
    ))

    if not tasks:
        log("DONE", "신규/개정 수집할 데이터가 없습니다. 종료.")
        return

    log("FETCH", f"동시 요청 수: {MAX_CONCURRENT} | 총 요청 대상: {len(tasks):,} 개")

    # -- asyncio 실행 --------------------------------------------------------
    records, tag_counter = asyncio.run(collect_all(tasks))
    log("FETCH", f"API 응답 성공: {len(records):,} 건")
    log("FETCH", "상태별 상세: " + ", ".join(
        f"{tag}={cnt:,}"
        for tag, cnt in sorted(tag_counter.items(), key=lambda x: -x[1])
    ))

    # DB 저장 (UPSERT)
    processed = save_to_db(records, engine)

    elapsed = time.time() - t0
    log("DONE", (
        f"DB 반영 시도: {processed:,} 건 (신규 삽입 + 개정 갱신 합산. "
        f"MySQL 특성상 값이 실제로 바뀌지 않은 행은 영향 0건으로 집계될 수 있음) | "
        f"소요시간: {elapsed:.1f} 초 ({elapsed / 60:.1f} 분)"
    ))


# -- 스크립트 직접 실행 시 ----------------------------------------------------
if __name__ == "__main__":
    main()


경로 설정 완료
  프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
  DATA 폴더     : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
  현재 위치     : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\collect\us_trade_export_data
  운영체제      : nt

[START] 미국 수출 데이터 수집 시작 (V3) | 2020-01 ~ 2026-05
[INFO] 수집 대상 HS 코드: 500 개
[INFO] 개정 반영 모드(REVISION_MODE): True (최근 3년치 강제 재수집 여부)
[DB] 테이블 'us_export_data' 준비 완료
[DB] 기존 저장 건수: 60,094 개
[INFO] 전체 가능 조합: 38,500 개 | 기존 보유: 60,094 개 | 신규 수집 대상: 723 개 | 개정 재수집 대상: 17,500 개 (기준월 >= 2023-07)
[FETCH] 동시 요청 수: 25 | 총 요청 대상: 18,223 개
수출 데이터 수집: 100%|████████████████████████████| 18223/18223 [04:14<00:00, 71.72req/s]
[FETCH] API 응답 성공: 17,500 건
[FETCH] 상태별 상세: success=17,500, no_data_204=723
[DONE] DB 반영 시도: 17,500 건 (신규 삽입 + 개정 갱신 합산. MySQL 특성상 값이 실제로 바뀌지 않은 행은 영향 0건으로 집계될 수 있음) | 소요시간: 256.1 초 (4.3 분)
